# BP3 — "Try On Your Own" Solutions (Tasks 1–3)

This is a **self-contained** solution directory for the three exercises at the end of
`day1_input_files/running_bp3.ipynb`. 

## Setup: imports and MPI environment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
%env OMPI_MCA_btl_vader_single_copy_mechanism=none
%env OMP_NUM_THREADS=1

In [ ]:
!chmod +x ../../NDP_helper_scripts/*

In [ ]:
tandem_exc = 'tandem_2d_6p'
mpi_rank = 30
NDP_mpi_cmds = "--mca btl_vader_single_copy_mechanism none --mca btl_vader_backing_directory /tmp --bind-to none"
NDP_mpi_wrapper = "../../NDP_helper_scripts/wrapper.sh"

## Baseline run

The unmodified BP3 scenario, run to `final_time = 7e9 s` over the first earthquake cycle.
Output probes are written with the `baseline_fltst_` prefix. This is the reference each
task is compared against.

In [ ]:
! mpiexec $NDP_mpi_cmds -n $mpi_rank $NDP_mpi_wrapper $tandem_exc bp3_baseline.toml --petsc -ts_monitor -options_file petsc_config.cfg

In [ ]:
sec_to_years = 356 * 24 * 3600
probe_15km = pd.read_csv('baseline_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='baseline')
plt.yscale('log')
plt.legend()
plt.xlabel('Time (years)')
plt.ylabel('Slip rate (m/s)')

## Task 1 — Change the loading rate

**What changed:** in `bp3_task1.lua` we added a new scenario
`bp3_d60_normal_2Vp = BP3.new{dip=60, Vp=-2e-9}` — the same geometry and friction as
`bp3_d60_normal` but with **double** the plate convergence rate (`Vp`: `-1e-9` → `-2e-9` m/s).
`bp3_task1.toml` selects this scenario and writes the `task1_fltst_` probes.

The loading rate enters only the Dirichlet boundary forcing, not the elastic operator, so the
precomputed `gf/gf_res_0.50` Green's functions are reused. Faster loading is expected to
**shorten the interseismic interval** and advance the first earthquake.

In [ ]:
! mpiexec $NDP_mpi_cmds -n $mpi_rank $NDP_mpi_wrapper $tandem_exc bp3_task1.toml --petsc -ts_monitor -options_file petsc_config.cfg

In [ ]:
sec_to_years = 356 * 24 * 3600
probe_15km = pd.read_csv('baseline_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='baseline')
probe_15km = pd.read_csv('task1_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task1')
plt.yscale('log')
plt.legend()
plt.xlabel('Time (years)')
plt.ylabel('Slip rate (m/s)')

## Task 2 — Change frictional properties

**What changed:** in `bp3_task2.lua` (scenario back to `bp3_d60_normal`) we modified the
rate-and-state friction:
- evolution-effect parameter `b`: `0.015` → `0.019` (larger `b - a`, stronger weakening)
- critical slip distance `D_c` (`BP3:L`): `0.008` → `0.012 m` (larger nucleation size,
  longer recurrence)

Friction does not enter the elastic operator, so `gf/gf_res_0.50` is reused. Output uses the
`task2_fltst_` prefix.

In [ ]:
! mpiexec $NDP_mpi_cmds -n $mpi_rank $NDP_mpi_wrapper $tandem_exc bp3_task2.toml --petsc -ts_monitor -options_file petsc_config.cfg

In [ ]:
sec_to_years = 356 * 24 * 3600
probe_15km = pd.read_csv('baseline_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='baseline')
probe_15km = pd.read_csv('task1_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task1')
probe_15km = pd.read_csv('task2_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task2')
plt.yscale('log')
plt.legend()
plt.xlabel('Time (years)')
plt.ylabel('Slip rate (m/s)')

## Task 3 — Vary shear modulus between the plates

**What changed:** in `bp3_task3.lua` the shear modulus `BP3:mu(x, y)` now returns a different
value on each side of the dipping fault — `mu_overriding = 30 GPa` (hanging wall) and
`mu_subducting = 25 GPa` (foot wall) — selected by the sign of `dist = x + y/tan(dip)`.
The Lamé parameter `lam` follows from `mu`, so the elastic operator is bimaterial.

Because the elastic properties changed, the **Green's functions must be recomputed**.
`bp3_task3.toml` points `gf_checkpoint.prefix` at a fresh `gf/gf_hetero` directory, so the
first stage rebuilds the operator before time stepping. Output uses the `task3_fltst_` prefix.

> This run is the longest of the three because it regenerates `G` from scratch.

In [ ]:
! mpiexec $NDP_mpi_cmds -n $mpi_rank $NDP_mpi_wrapper $tandem_exc bp3_task3.toml --petsc -ts_monitor -options_file petsc_config.cfg

In [ ]:
sec_to_years = 356 * 24 * 3600
probe_15km = pd.read_csv('baseline_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='baseline')
probe_15km = pd.read_csv('task1_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task1')
probe_15km = pd.read_csv('task2_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task2')
probe_15km = pd.read_csv('task3_fltst_dp150.csv', comment='#')
plt.plot(probe_15km['Time'] / sec_to_years, np.abs(probe_15km['slip-rate0']), label='task3')
plt.yscale('log')
plt.xlabel('Time (years)')
plt.legend()
plt.ylabel('Slip rate (m/s)')